## 类型提示 (Type Hints)

版本

- Python 3.5+ 基础类型提示 `x: int`、`-> int`
- Python 3.9+ 内置容器可直接标注（如 `list[int]`）
- Python 3.10+ 支持 `X | Y`
- Python 3.12+ 支持 `type` 语句定义类型别名


总结
- 类型提示运行时不做任何事，它的用户是静态检查器和 IDE。
- 函数签名：`def f(x: int) -> str`:；变量：`x: int = 0`。
- 容器直接用内置类型：`list[str]`、`dict[str, int]`，不需要 `typing.List`。
- 联合用 `|`：`int | str`、`int | None`，`Optional` 和 `Union` 已是遗留写法。
- 泛型用 `TypeVar`（旧版）或 `[T]` 内联语法（3.12+）
- 固定键的字典用 `TypedDict`，结构化接口用 `Protocol`。
- 实践原则：渐进式添加，避免滥用 `Any`，`type: ignore` 带上错误码

### 一、什么是类型提示？

类型提示（Type Hints）是 Python 3.5+ 引入的`静态类型标注机制`，用于在代码中标明变量、参数、返回值的预期类型。

In [2]:
# ❌ 没有类型提示 —— 不知道参数和返回值是什么类型
def process(data, flag):
    if flag:
        return data.upper()
    return data

# ✅ 有类型提示 —— 一目了然
def process(data: str, flag: bool) -> str:
    if flag:
        return data.upper()
    return data

### 二、基础类型标注

In [9]:
# 基本语法：变量名: 类型 = 值
name: str = "Alice"
age: int = 25
height: float = 1.75
is_student: bool = True
data: bytes = b"raw"     # 字节
nothing: None = None     # None 类型

In [10]:
# ⚠️ 类型提示不会阻止赋值错误（运行时不检查）
x: int = "hello"

# ✅ 只声明不赋值也可以（用于类属性等场景）
name: str  # 仅声明类型，未赋值

### 三、容器类型标注

#### 3.1 Python 3.9+ 推荐写法（内置泛型）

In [11]:
# 列表
names: list[str] = ["Alice", "Bob"]
scores: list[int] = [90, 85, 92]

# 字典
user: dict[str, int] = {"age": 25, "score": 100}

# 元组（固定长度和类型）
point: tuple[float, float] = (3.14, 2.71)

# 元组（可变长度，同一类型）
numbers: tuple[int, ...] = (1, 2, 3, 4, 5)

# 集合
tags: set[str] = {"python", "typing"}

# 冻结集合
ids: frozenset[int] = frozenset({1, 2, 3})


#### 3.2 Python 3.8 及以下（需从 typing 导入）

In [12]:
from typing import List, Dict, Tuple, Set

names: List[str] = ["Alice", "Bob"]
user: Dict[str, int] = {"age": 25}
point: Tuple[float, float] = (3.14, 2.71)
tags: Set[str] = {"python"}

#### 3.3 嵌套容器

In [14]:
# 字典的值是列表
students: dict[str, list[int]] = {
    "Alice": [90, 85, 92],
    "Bob": [78, 88, 95]
}

# 列表中包含字典
records: list[dict[str, str]] = [
    {"name": "Alice", "city": "Beijing"},
    {"name": "Bob", "city": "Shanghai"}
]

### 四、函数类型标注

默认参数

In [ ]:
def greet(name: str, greeting: str = "Hello") -> str:
    return f"{greeting}, {name}!"

*args 和 **kwargs

In [ ]:
def total(*args: int) -> int:
    return sum(args)

def config(**kwargs: str) -> None:
    for key, value in kwargs.items():
        print(f"{key} = {value}")

多个返回值

In [15]:
# 使用 tuple 标注
def divide(a: int, b: int) -> tuple[int, int]:
    return a // b, a % b

quotient, remainder = divide(10, 3)

### 五、联合类型：Union、Optional、Callable

#### 5.1 Union


`T1 | T2` 等价于 Union[T1, T2]

In [ ]:
from typing import Union

# Python 3.10+ 推荐写法
def process(value: int | str) -> str:
    return str(value)

# Python 3.9 及以下
def process(value: Union[int, str]) -> str:
    return str(value)

#### 5.2 Optional

`T | None` 等价于 Optional[T]

In [19]:
from typing import Optional
# 以下两种写法等价：
def find_user(name: str) -> Optional[str]:  # 等价于 str | None
    pass

def find_user(name: str) -> str | None:     # Python 3.10+
    pass

#### 5.3 Callable

标注函数或可调用对象：`Callable[[参数类型], 返回类型]`

In [17]:
from typing import Callable

# Callable[[参数类型1, 参数类型2, ...], 返回类型]
def apply(func: Callable[[int, int], int], a: int, b: int) -> int:
    return func(a, b)

result = apply(lambda x, y: x + y, 3, 4)  # 7
result

7

#### 5.4 其他
- Any —— 任意类型
- Literal —— 限定具体值
- Final —— 不可重新赋值

### 六、类型别名

TypeAlias与type

In [ ]:
# 旧写法
from typing import TypeAlias
Vector: TypeAlias = list[float]

# 新写法（3.12+）
type Vector = list[float]

### 七、泛型

泛型函数

In [ ]:
# 旧写法（3.12 之前）
from typing import TypeVar, Callable
T = TypeVar("T")
U = TypeVar("U")

def zip_map(a: list[T], b: list[U], f: Callable[[T, U], T]) -> list[T]:
    return [f(x, y) for x, y in zip(a, b)]

# 新写法（3.12+）
def zip_map[T, U](a: list[T], b: list[U], f: Callable[[T, U], T]) -> list[T]:
    return [f(x, y) for x, y in zip(a, b)]

泛型类

In [ ]:
# 旧写法
from typing import Generic, TypeVar
T = TypeVar("T")

class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, item: T) -> None:
        self._items.append(item)
    def pop(self) -> T:
        return self._items.pop()

# 新写法（3.12+）
class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, item: T) -> None:
        self._items.append(item)
    def pop(self) -> T:
        return self._items.pop()

### 八、结构化

Protocol-结构化子类

In [24]:
# Protocol 实现了鸭子类型的静态检查版本——不需要显式继承，只要实现了指定方法就算符合。
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> None: ...

class Circle:
    def draw(self) -> None:
        print("Drawing a circle")

class Square:
    def draw(self) -> None:
        print("Drawing a square")

def render(shape: Drawable) -> None:
    shape.draw()

render(Circle())   # ✅ Circle 实现了 draw 方法
render(Square())   # ✅ Square 实现了 draw 方法
# render("hello")    # ❌ str 没有 draw 方法

Drawing a circle
Drawing a square


TypedDict-结构化字典

In [22]:
from typing import TypedDict

class User(TypedDict):
    name: str
    age: int
    email: str

def create_user(name: str, age: int, email: str) -> User:+
    return {"name": name, "age": age, "email": email}

user: User = {"name": "Alice", "age": 25, "email": "a@b.com"}
user["name"]   # ✅ mypy 知道这是 str
# user["foo"]    # ❌ mypy 报错：没有这个键

'Alice'

### 九、最佳实践与常见陷阱

最佳实践

In [ ]:
# ✅ 1. 公共函数/方法一定要标注
def calculate_area(radius: float) -> float:
    return 3.14159 * radius ** 2

# ✅ 2. 复杂类型用 TypeAlias 简化
Matrix: TypeAlias = list[list[float]]

# ✅ 3. 用 Protocol 代替 ABC（当不需要继承关系时）
class Serializable(Protocol):
    def to_json(self) -> str: ...

# ✅ 4. 善用 Literal 限制取值范围
def set_color(color: Literal["red", "green", "blue"]) -> None: ...

# ✅ 5. 类属性要标注
class User:
    name: str
    age: int

    def __init__(self, name: str, age: int) -> None:
        self.name = name
        self.age = age


常见陷阱

In [ ]:
# ❌ 陷阱 1：可变默认参数
def bad(items: list[int] = []) -> list[int]:  # 危险！
    items.append(1)
    return items

# ✅ 正确写法
def good(items: list[int] | None = None) -> list[int]:
    if items is None:
        items = []
    items.append(1)
    return items

# ❌ 陷阱 2：混淆 Optional 和默认值
def search(query: str, limit: int = None) -> list[str]:  # ❌
    ...

def search(query: str, limit: int | None = None) -> list[str]:  # ✅
    ...

# ❌ 陷阱 3：Any 传染
def get_data() -> Any:  # 调用者拿到的也是 Any，类型信息丢失
    ...

# ✅ 尽量返回具体类型
def get_data() -> dict[str, str]:
    ...
